# Notebook A — Baseline Federated Frameworks (B1, B3, B4)

This notebook runs the three baselines needed to reproduce **Table 2** and to feed
**Tables 3 and 6** of the BiAB-IoT manuscript:

* **B1 — Centralised Isolation Forest**: single-node IF trained on the whole
  training set, no federation.
* **B3 — FL with local anomaly filtering**: each gateway locally excludes
  its own high-anomaly-score clients before contributing; no shared trust
  state, no consensus.
* **B4 — FLIT-style reputation-weighted FL**: reputation is updated each
  round from client contribution quality and is used to weight aggregation.
  No on-chain policy triggering (P1); this is what BiAB-IoT extends.

**Prerequisites.** Run `BIAB.ipynb` once first to produce `binary_dataset.pkl`
in `/content/drive/MyDrive/`. This notebook loads that pickle and reuses the
same train/test split and client partition.

Results are persisted to `/content/drive/MyDrive/path1_results/` for
`Notebook_D_Aggregate.ipynb` to pick up.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install --upgrade scikit-learn tensorflow
import os, sys
# Put biab_common.py next to this notebook on Drive, then:
sys.path.insert(0, '/content/drive/MyDrive/path1_code')
from biab_common import (
    load_dataset, make_clients, poison_clients, make_model, train_local,
    fed_average, evaluate, save_result, seed_everything, SEED, RESULT_DIR,
)
seed_everything(SEED)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
data = load_dataset()
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
input_dim = X_train.shape[1]
print('X_train', X_train.shape, 'X_test', X_test.shape, 'dim', input_dim)

## B1 — Centralised Isolation Forest

An IF is trained on the *benign portion only* of the training set (this is
how Isolation Forest is typically used for anomaly detection: the model
learns the normal manifold, then anomalies are anything with a short average
path length). Contamination is set to the empirical attack ratio (~18%),
matching the manuscript's Simulation Configuration section.

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

CONTAMINATION = 0.18   # matches CIC-IoT-2023 empirical attack ratio
N_ESTIMATORS = 100

clf = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=SEED,
    n_jobs=-1,
)
# IF learns 'normal' — fit on benign only
clf.fit(X_train[y_train.values == 0])

# IF: -1 = anomaly, +1 = normal. Convert to our label: 1 = attack, 0 = benign.
raw = clf.predict(X_test)
pred = (raw == -1).astype(int)

from biab_common import evaluate
class _StaticModel:
    def __init__(self, p): self.p = p
    def predict(self, X, verbose=0):
        return self.p.reshape(-1, 1)   # match make_model output shape
metrics_b1 = evaluate(_StaticModel(pred), X_test, y_test)
print('B1 (Centralised IF):', metrics_b1)
save_result('B1_centralised_if', metrics_b1,
            {'n_estimators': N_ESTIMATORS, 'contamination': CONTAMINATION})

## B3 — FL with local anomaly filtering (no blockchain)

Each gateway trains a *local* Isolation Forest on its own clients' feature
distributions and drops the top-anomaly-scoring clients from that round's
contribution.  Exclusion is stateless: a client excluded in round *r* is
free to contribute again in round *r+1*.  This is the local-only defence
that the manuscript's B3 row represents.

We evaluate at the manuscript's default 10% attack condition.

In [ ]:
import numpy as np
import random

NUM_CLIENTS = 500
CLIENTS_PER_ROUND = 50
ROUNDS = 10
ATTACK_PCT = 0.10           # matches manuscript B3 evaluation point
LOCAL_ANOMALY_THRESHOLD = 0.7   # exclude top-30% anomaly-score clients each round

seed_everything(SEED)
clients = make_clients(X_train, y_train, NUM_CLIENTS, SEED)
poisoned, mal_ids = poison_clients(clients, ATTACK_PCT, SEED)

def client_anomaly_score(Xc):
    # Stateless: each round, fit a mini-IF on the round's pool
    from sklearn.ensemble import IsolationForest
    m = IsolationForest(n_estimators=50, contamination='auto',
                         random_state=SEED, n_jobs=-1)
    m.fit(Xc[:2000])
    # avg negative decision function -> higher = more anomalous
    return -np.mean(m.decision_function(Xc[:2000]))

global_model = make_model(input_dim)
global_weights = global_model.get_weights()

for rnd in range(ROUNDS):
    pool = random.sample(list(enumerate(poisoned)), CLIENTS_PER_ROUND)
    # Rank by local anomaly score, drop top ~30%
    scored = [(cid, client_anomaly_score(Xc), Xc, yc) for cid, (Xc, yc) in pool]
    scored.sort(key=lambda t: t[1], reverse=True)   # most anomalous first
    n_keep = int(round(CLIENTS_PER_ROUND * LOCAL_ANOMALY_THRESHOLD))
    kept = scored[-n_keep:]                         # keep the least anomalous
    local_ws = [train_local(Xc, yc, global_weights, input_dim) for _, _, Xc, yc in kept]
    global_weights = fed_average(local_ws)
    global_model.set_weights(global_weights)
    print(f'  round {rnd+1}: kept {n_keep}/{CLIENTS_PER_ROUND}')

metrics_b3 = evaluate(global_model, X_test, y_test)
print('B3 (FL + local anomaly):', metrics_b3)
save_result('B3_fl_local_anomaly', metrics_b3,
            {'attack_pct': ATTACK_PCT, 'rounds': ROUNDS,
             'local_anomaly_threshold': LOCAL_ANOMALY_THRESHOLD})

## B4 — FLIT-style reputation-weighted FL

FLIT (Das et al. 2025, ref [19] in the manuscript) uses on-chain reputation
to weight federated aggregation but does not implement AI→Blockchain policy
triggering.  We implement the reputation update rule as:

* Every client starts with reputation `1.0`.
* After each round, a client's reputation is updated based on **cosine
  similarity** between its local weight-delta and the aggregated global
  delta.  Clients whose direction aligns with the aggregate keep or gain
  reputation; clients whose direction is orthogonal or opposite lose it.
* Reputation is clipped to `[0.0, 1.0]`.
* Aggregation is reputation-weighted (uses `fed_average(..., weights_scale=)`).
* **No exclusion**: even low-reputation clients still contribute, just with
  smaller weight. That's the key difference from BiAB-IoT (P1).

Same 10% attack condition as B3 so the two rows are directly comparable.

In [ ]:
import numpy as np

REPUTATION_LR = 0.15  # how fast reputation moves per round

def flatten_weights(ws):
    return np.concatenate([w.flatten() for w in ws])

seed_everything(SEED)
clients = make_clients(X_train, y_train, NUM_CLIENTS, SEED)
poisoned, mal_ids = poison_clients(clients, ATTACK_PCT, SEED)
reputation = np.ones(NUM_CLIENTS, dtype=np.float64)

global_model = make_model(input_dim)
global_weights = global_model.get_weights()

for rnd in range(ROUNDS):
    pool = random.sample(range(NUM_CLIENTS), CLIENTS_PER_ROUND)
    prev_flat = flatten_weights(global_weights)
    local_ws = []
    for cid in pool:
        Xc, yc = poisoned[cid]
        w = train_local(Xc, yc, global_weights, input_dim)
        local_ws.append(w)

    round_reps = np.array([reputation[cid] for cid in pool])
    new_global = fed_average(local_ws, weights_scale=round_reps)
    new_flat = flatten_weights(new_global)
    agg_delta = new_flat - prev_flat

    # Update reputation based on alignment with aggregate delta
    for j, cid in enumerate(pool):
        client_delta = flatten_weights(local_ws[j]) - prev_flat
        num = np.dot(client_delta, agg_delta)
        den = (np.linalg.norm(client_delta) * np.linalg.norm(agg_delta)) + 1e-12
        cos = num / den
        reputation[cid] = np.clip(
            reputation[cid] + REPUTATION_LR * cos, 0.0, 1.0)

    global_weights = new_global
    global_model.set_weights(global_weights)
    caught = np.mean([reputation[cid] < 0.5 for cid in mal_ids])
    print(f'  round {rnd+1}: mean reputation of malicious clients ='
          f' {np.mean([reputation[cid] for cid in mal_ids]):.3f}, '
          f'{caught*100:.1f}% below 0.5')

metrics_b4 = evaluate(global_model, X_test, y_test)
print('B4 (FLIT):', metrics_b4)
save_result('B4_flit', metrics_b4,
            {'attack_pct': ATTACK_PCT, 'rounds': ROUNDS,
             'reputation_lr': REPUTATION_LR})

## Summary — three baseline rows for Table 2

Once this notebook completes, three result files exist in
`/content/drive/MyDrive/path1_results/`:

* `B1_centralised_if.pkl`
* `B3_fl_local_anomaly.pkl`
* `B4_flit.pkl`

`Notebook_D_Aggregate.ipynb` will combine these with the results from
`Notebook_B_BiAB_Sweep.ipynb` and `Notebook_C_PBFT_Benchmark.ipynb` into the
final Tables 2–6.